# 01 · OpenClaw from Python

OpenClaw is a Node.js agent runtime. Its **gateway** is the control plane: a WebSocket server that every
client (the CLI, the web UI, the mobile apps) talks to. The [`openclaw-sdk`](https://masteryodaa.github.io/openclaw-sdk/)
package is a Python client for that protocol, so from Python we can:
* run agent turns;
* stream what the agent is doing;
* read and write its workspace files.

The rest of this series builds on that. In this notebook there's no MLflow yet, just the SDK, so you can see
the raw material MLflow will be built on:

```
Python (this notebook)                                   OpenClaw gateway (Node)        Claude
  openclaw_sdk.OpenClawClient  ── WebSocket, protocol v4 ──>  agent "main"  ── Messages API ──>  claude-haiku-5-5
                               <── agent / chat events ────     workspace/AGENTS.md
                                                                workspace/knowledge/*.md
```

| Notebook | |
|---|---|
| **01 · OpenClaw from Python** (this one) | gateway, SDK client, a turn, its event stream, workspace files, session history |
| **02 · MLflow on top of OpenClaw** | turn those events into MLflow traces; AGENTS.md from the Prompt Registry |
| **03 · Hermes Agent from Python** | the same, for Nous Research's Hermes Agent, in-process |
| **04 · Evaluate both agents** | `mlflow.genai.evaluate` with code checks, Claude judges, and human labels |
| **05 · Improve and gate** | failures → Claude proposes a new AGENTS.md → evaluate on held-out data → promote or not |

## Setup

Everything in this series is isolated:
* MLflow writes to `notebooks/.mlflow/sdk.db`.
* OpenClaw gets its own state directory, config, token, and port (19556) under `notebooks/.mlflow/sdk/gateway/`.
* Your own `~/.openclaw` and any gateway you already run (port 18789) are never touched.

The model is Claude Haiku 5.5, through your `ANTHROPIC_API_KEY` in `.env`.

This notebook starts the series over. `fresh=True` deletes `sdk.db` (traces, prompt versions, evaluation runs),
and the workspaces get the flawed v1 `AGENTS.md` again.

In [1]:
from sia import notebook as nb
from sia.workspace import seed_agents_md

nb.setup(db="sdk.db", experiment="acme-support-sdk", fresh=True)
for runtime in ("openclaw", "hermes"):
    nb.sdk_workspace(runtime, agents_md=seed_agents_md())

sia.config was already imported in this kernel; restart the kernel to change settings.


tracking_uri  http://127.0.0.1:5000
experiment    acme-support-agent (id 1)
backend       openclaw
agent_model   anthropic:/claude-haiku-5-5
judge_model   anthropic:/claude-opus-5-5
llm_judges    True


In [2]:
import importlib.metadata
import shutil
import subprocess

from sia import config

assert shutil.which(config.OPENCLAW_BIN), "install OpenClaw first: npm i -g openclaw"
print("OpenClaw    ", subprocess.run([config.OPENCLAW_BIN, "--version"], capture_output=True, text=True).stdout.strip())
print("openclaw-sdk", importlib.metadata.version("openclaw-sdk"))
print("agent model ", config.AGENT_MODEL)

OpenClaw     OpenClaw 2026.10.5 (1b58cb9)
openclaw-sdk 2.1.0
agent model  anthropic:/claude-haiku-5-5


## 1. A workspace

An OpenClaw agent's behavior is plain files in its workspace:
* `AGENTS.md` holds its instructions, always in context;
* `knowledge/` holds the policy files it can open with the `read` tool.

This `AGENTS.md` is the scenario's deliberately flawed v1. It has stale facts, a mandatory disclaimer, and
"give your best guess".

In [3]:
ws = nb.sdk_workspace("openclaw")
print(ws)
print(sorted(p.name for p in (ws / "knowledge").iterdir()))
print()
print((ws / "AGENTS.md").read_text())

/Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk/openclaw
['account_access.md', 'billing.md', 'product_howto.md', 'support_process.md']

# Acme Cloud Support Agent

You are the support agent for Acme Cloud. You answer customer questions about
billing, account access, and how to use the product.

## How to answer
- Be warm and thorough. Start by reminding the customer that policies can vary by
  plan and region and that they should always double-check the Terms of Service.
- Answer quickly from the quick facts below. Looking things up slows you down, so
  only open files if you really have to.
- Keep the customer happy. If you are not sure, give your best guess.

## Quick facts
- Refunds: customers can get a refund within 30 days of purchase.
- Password reset links are valid for 24 hours.
- Data export: Settings → Data → Export.
- Teammates: Workspace → Members → Invite.



## 2. A gateway of our own

`Gateway` writes an OpenClaw config and runs `openclaw gateway run` as a subprocess. The config sets:
* this workspace;
* Claude Haiku 5.5;
* only the `read` tool;
* token auth;
* loopback only.

`start()` returns as soon as the gateway logs `ready`, and reuses the gateway if this directory already started
one.

In [4]:
from sia.openclaw_client import Gateway

gw = Gateway(nb.NB_STORE / "sdk" / "gateway", nb.sdk_workspace("openclaw")).start()   # reuses a running one
print(gw.url, "· pid", gw.pid(), "· workspace", gw.workspace)

ws://127.0.0.1:19556 · pid 30863 · workspace /Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk/openclaw


In [5]:
import json

cfg = json.loads(gw.config_path.read_text())
cfg["gateway"]["auth"]["token"] = "<redacted>"
print(json.dumps(cfg, indent=2))

{
  "agents": {
    "defaults": {
      "workspace": "/Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk/openclaw",
      "model": {
        "primary": "anthropic/claude-haiku-5-5"
      },
      "skipBootstrap": true
    }
  },
  "tools": {
    "allow": [
      "read"
    ]
  },
  "gateway": {
    "mode": "local",
    "port": 19556,
    "bind": "loopback",
    "auth": {
      "mode": "token",
      "token": "<redacted>"
    }
  },
  "plugins": {
    "entries": {
      "bonjour": {
        "enabled": false
      }
    }
  }
}


## 3. Connect with openclaw-sdk

The SDK's entry point is `OpenClawClient`. One wrinkle: openclaw-sdk 2.1.0 speaks gateway protocol **v3**, and
OpenClaw 2026.9 gateways speak **v4**. `sia.openclaw_client.connect()` swaps in a gateway class that changes
only the `connect` handshake. It sends:
* protocol 4;
* the `tool-events` capability (without it the gateway doesn't stream tool calls to the client);
* the token and a device key from this gateway's directory, instead of `~/.openclaw`.

Everything after the handshake is the SDK's own code.

In [6]:
import inspect
from sia import openclaw_client

src = inspect.getsource(openclaw_client._sdk_gateway_class)
print(src[src.index("        async def _handle_challenge"):src.index("            req_id")])

        async def _handle_challenge(self, payload: dict[str, Any]) -> None:
            nonce = payload.get("nonce", "")
            signed_at = int(payload.get("ts") or time.time() * 1000)  # v4: sign the server's timestamp
            d = self._identity
            signature = protocol._sign_device_payload(d["privateKeyPem"], "|".join(
                ["v2", d["deviceId"], "cli", "cli", "operator", ",".join(SCOPES), str(signed_at), self._token, nonce]))
            params = {
                "minProtocol": PROTOCOL, "maxProtocol": PROTOCOL,
                "client": {"id": "cli", "version": protocol._SDK_VERSION, "platform": platform.system().lower(),
                           "mode": "cli"},
                "role": "operator", "scopes": SCOPES,
                "caps": ["tool-events"],  # without it the gateway doesn't stream tool calls to us
                "commands": [], "permissions": {}, "auth": {"token": self._token},
                "locale": "en-US", "userAgent": f"openclaw-

In [7]:
client = await openclaw_client.connect(gw)     # Jupyter runs top-level await on its own event loop
client

OpenClawClient(mode='auto', connected=True)

In [8]:
agents = await client.gateway.call("agents.list", {})
models = await client.gateway.call("models.list", {})
print("agents:", [(a["id"], a["workspace"]) for a in agents["agents"]])
print("models:", [(m["provider"], m["id"]) for m in models["models"]][:5])

agents: [('main', '/Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk/openclaw')]
models: [('anthropic', 'claude-haiku-5-5'), ('anthropic', 'claude-sonnet-5-5'), ('anthropic', 'claude-sonnet-5'), ('anthropic', 'claude-sonnet-4-6')]


## 4. One turn

`client.get_agent(agent_id, session_name=...)` is a handle on one conversation with one agent. A new session
name means a fresh conversation. `execute()` sends the message (`chat.send`) and collects the streamed events
until the run finishes.

In [9]:
import uuid

QUESTION = "Can we set up SSO with Okta on our Pro plan?"

agent = client.get_agent("main", session_name=f"nb01-{uuid.uuid4().hex[:8]}")
result = await agent.execute(QUESTION)
print(f"success={result.success}  latency={result.latency_ms} ms  stop_reason={result.stop_reason}\n")
print(result.content)

success=True  latency=19882 ms  stop_reason=stop

Policies can vary by plan and region, so please double-check the Terms of Service for the details that apply to you.

I don't have confirmed information on whether SSO with Okta is included in the Pro plan, so I can't say yes or no. Here's how to get a definitive answer:

- **Check your plan's feature list** on the Acme Cloud pricing page, or in your account's billing settings.
- **Contact Acme Cloud support or your account rep** and ask specifically whether SSO (SAML/OIDC) with Okta is available on Pro, and if not, what the upgrade path and cost look like.

If it turns out to be available, setup typically involves configuring a SAML or OIDC app in Okta and then entering the connection details in your Acme Cloud admin settings. Your Acme Cloud admin or support team can confirm the exact steps for your account.


## 5. What the agent did: the event stream

While a run is going, the gateway pushes events: `agent` events (one per stream) and `chat` events (state
changes). The SDK hands every one of them to `CallbackHandler.on_stream_event`. A tiny handler that just
records them shows the vocabulary we'll map to MLflow spans in notebook 02.

In [10]:
import pandas as pd
from openclaw_sdk.callbacks.handler import CallbackHandler


class EventLog(CallbackHandler):
    def __init__(self):
        self.events = []

    async def on_stream_event(self, agent_id, event):
        p = event.data.get("payload") or {}
        self.events.append({"event": event.data.get("event"), "stream": p.get("stream") or p.get("state"),
                            "ts": p.get("ts"), "data": p.get("data") or {}})


log = EventLog()
agent = client.get_agent("main", session_name=f"nb01-{uuid.uuid4().hex[:8]}")
result = await agent.execute("Please read knowledge/billing.md and tell me: what happens if my card payment fails?",
                             callbacks=[log])
print(result.content, "\n")
pd.DataFrame(log.events).groupby(["event", "stream"]).size().rename("events")

If a card payment fails, Acme Cloud retries it 3 times over 7 days. If the third attempt fails, the workspace is downgraded to the Free plan. No data is deleted.

I didn't check the billing contact or invoice settings for this, so if you need to update your card or billing email, Settings → Billing → Billing contact is where to do it.

Also, the quick-facts section of my support instructions says refunds are available within 30 days. The billing policy file, last updated July 1, 2026, says 14 days for all plans. I'd go with the 14-day policy, since it's the more recent and more specific source. 



event  stream   
agent  assistant    2
       thinking     2
       tool         2
chat   delta        2
       final        1
Name: events, dtype: int64

The `tool` stream is what tells us *how* the answer was found. Each tool call is a `start` event (name and
arguments) and a `result` event, paired by `toolCallId` and stamped with the gateway's own timestamps.

v1's instructions tell the agent to answer from its "quick facts" and avoid opening files. Left to itself, it
doesn't open any (and v1 never even mentions `knowledge/`): the SSO answer above is a guess, not a read of
`account_access.md`. That's the failure notebooks 04 and 05 measure and fix. Here the question names the policy
file, so there's a tool call to look at.

(openclaw-sdk also has `on_tool_call` / `on_tool_result` hooks, but they expect protocol v3's `call` phase, so
on a v4 gateway they never fire. `on_stream_event` sees everything.)

In [11]:
tools = [e for e in log.events if e["stream"] == "tool"]
pd.DataFrame([{"phase": e["data"].get("phase"), "tool": e["data"].get("name"),
               "call": e["data"].get("toolCallId", "")[-8:],
               "args / result": e["data"].get("args") or e["data"].get("meta"),
               "ts": e["ts"]} for e in tools], columns=["phase", "tool", "call", "args / result", "ts"])

,phase,tool,call,args / result,ts
0,start,read,C32EJqpA,{'path': 'knowledge/billing.md'},1792495698874
1,result,read,C32EJqpA,from knowledge/billing.md,1792495698939


## 6. The workspace, through the gateway

The gateway also exposes the agent's workspace files (`agents.files.list/get/set`), and the SDK wraps them as
`list_files()`, `get_file()`, and `set_file()`. That's the **update** hook for the rest of the series: a new
`AGENTS.md` version goes into the running agent with one `set_file` call. No files are copied by hand, and the
gateway doesn't restart.

In [12]:
files = await agent.list_files()
pd.DataFrame(files["files"])[["name", "missing", "size"]]

,name,missing,size
0,AGENTS.md,False,754.0
1,SOUL.md,True,NaN
2,USER.md,True,NaN
3,BOOTSTRAP.md,True,NaN
4,MEMORY.md,True,NaN


In [13]:
current = (await agent.get_file("AGENTS.md"))["file"]["content"]
print(current[:400], "...")

# Acme Cloud Support Agent

You are the support agent for Acme Cloud. You answer customer questions about
billing, account access, and how to use the product.

## How to answer
- Be warm and thorough. Start by reminding the customer that policies can vary by
  plan and region and that they should always double-check the Terms of Service.
- Answer quickly from the quick facts below. Looking things  ...


## 7. What it cost: session history

The streamed events say what happened, but not what it cost. The session transcript (`chat.history`) records
every model turn with:
* the model;
* token usage, including prompt-cache reads and writes;
* cost;
* the stop reason;
* the tool calls requested.

Notebook 02 uses it for the LLM spans.

In [14]:
history = (await client.gateway.call("chat.history", {"sessionKey": agent.session_key}))["messages"]
pd.DataFrame([{
    "role": m["role"], "model": m.get("model", ""), "stop": m.get("stopReason", ""),
    "tool calls": [b["name"] for b in m.get("content", []) if isinstance(b, dict) and b.get("type") == "toolCall"],
    "total tokens": (m.get("usage") or {}).get("totalTokens"),
    "out tokens": (m.get("usage") or {}).get("output"),
    "cost $": round(((m.get("usage") or {}).get("cost") or {}).get("total", 0), 5),
} for m in history])

,role,model,stop,tool calls,total tokens,out tokens,cost $
0,user,,,[],NaN,NaN,0.00000
1,assistant,claude-haiku-5-5,toolUse,[read],8675.0,86.0,0.00033
2,toolResult,,,[],NaN,NaN,0.00000
3,assistant,claude-haiku-5-5,stop,[],9410.0,434.0,0.00035


In [15]:
await client.close()     # the gateway keeps running for the next notebooks

**Next:** 02 · MLflow on top of OpenClaw. The same events, as MLflow traces.